# Harnesses: del modelo al agente

**Modelos Generativos Profundos · MIA · ICAI · 2026-2027**

En este tutorial usamos un modelo local (Qwen3.5-4B) como cerebro de tres harnesses:
**DeepSeek Harness**, **Codex** y **Claude Code**. Nos centramos en DeepSeek Harness.

Recorrido:

1. Una llamada al modelo (sin harness).
2. Un harness mínimo escrito a mano.
3. DeepSeek Harness: sus tools y una tool nuestra que genera documentos Word.
4. Codex y Claude Code sobre el mismo modelo (opcional).
5. Weight-space programming: leer y editar pesos de Qwen.

Todo el código está en `src/harness_lab/`. Aquí solo lo usamos.

## 0. Preparación

En la DGX, en la terminal de code-server, desde la carpeta del repositorio:

```bash
source scripts/dgx_env.sh   # cachés en clusters/dgx (cada sesión)
uv sync                     # instala todo en .venv
```

Después abrid este notebook y elegid el kernel `.venv/bin/python`.

> Sin GPU podéis ensayar todo con un modelo simulado: `export HARNESS_SCRIPTED=1` antes de abrir
> el notebook. Las respuestas son fijas, pero el flujo es el mismo.

In [ ]:
import json, os, shutil
from pathlib import Path

import torch
from harness_lab import config

print("Modelo:      ", config.MODEL_ID)
print("Simulado:    ", config.SCRIPTED)
print("GPU:         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no")
print("Workspace:   ", config.WORKSPACE_DIR)

## 1. El modelo

Qwen3.5-4B ocupa unos 9,3 GB en bf16. Cabe en la GPU de 16 GB de la DGX y deja sitio para
la caché KV. La primera descarga tarda unos minutos.

In [ ]:
from harness_lab.download import download

if not config.SCRIPTED:
    download(config.MODEL_ID)

Arrancamos el servidor local. Es un proceso aparte que carga el modelo una vez y ofrece
**tres APIs** distintas sobre el mismo modelo:

| Endpoint | API | Quién la usa |
|---|---|---|
| `/v1/chat/completions` | OpenAI Chat Completions | nuestras llamadas y el harness mínimo |
| `/v1/messages` | Anthropic Messages | DeepSeek Harness y Claude Code |
| `/v1/responses` | OpenAI Responses | Codex |

Código: `src/harness_lab/server/`. Cada petición se guarda en `outputs/traffic/`.

In [ ]:
from harness_lab.server.launch import start_server

server = start_server()   # unos 30-60 s la primera vez

## 2. Una llamada al modelo (sin harness)

Texto entra, texto sale. El modelo no recuerda nada entre llamadas y no ejecuta nada.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=config.BASE_URL, api_key=config.API_KEY)
reply = client.chat.completions.create(
    model=config.MODEL_ID,
    messages=[{"role": "user", "content": "Explica en una frase qué es un agente de IA."}],
)
print(reply.choices[0].message.content)

Ahora le damos una tool. El modelo **no la ejecuta**: solo pide que alguien la ejecute.
El servidor convierte el texto `<tool_call>...` del modelo en JSON (`server/toolcalls.py`).

In [ ]:
tools = [{
    "type": "function",
    "function": {
        "name": "create_word_document",
        "description": "Create a Word document from Markdown",
        "parameters": {
            "type": "object",
            "properties": {"filename": {"type": "string"}, "markdown": {"type": "string"}},
            "required": ["filename", "markdown"],
        },
    },
}]
reply = client.chat.completions.create(
    model=config.MODEL_ID,
    messages=[{"role": "user", "content": "Crea un documento Word con tres ideas sobre agentes."}],
    tools=tools,
)
msg = reply.choices[0].message
print("finish_reason:", reply.choices[0].finish_reason)
for call in msg.tool_calls or []:
    print(call.function.name, json.loads(call.function.arguments))

El modelo ha pedido la tool, pero no existe ningún fichero. Falta alguien que:
ejecute la tool, devuelva el resultado al modelo y repita hasta terminar.
**Eso es el harness.**

## 3. Un harness mínimo

`src/harness_lab/mini_harness.py` tiene unas 100 líneas. El bucle es:

1. Llamar al modelo con el historial y las tools.
2. Si no pide tools, terminar.
3. Si pide tools, ejecutarlas, añadir los resultados al historial y volver a 1.

In [ ]:
from harness_lab.mini_harness import MiniHarness

harness = MiniHarness()
print("Tools:", list(harness.registry))
answer = harness.run(
    "Crea un documento Word llamado agentes.docx con un título y tres ideas sobre agentes. "
    "Después léelo para comprobarlo."
)
print("\nRespuesta:", answer)

In [ ]:
# El historial completo: esto es lo que el modelo ve en cada vuelta
for m in harness.messages:
    print(f"{m['role']:>9}: {str(m.get('content') or m.get('tool_calls'))[:150]}")

## 4. DeepSeek Harness

DeepSeek Harness (`dsh`, agosto 2026, MIT) es el harness abierto de DeepSeek. Idea central:
**todo es un plugin** (modelos, tools, skills, sesiones, sandbox, compactación).
Está escrito en TypeScript, pero `uv sync` ya instaló el SDK de Python con su runtime.

Su adaptador de DeepSeek habla la API Messages de Anthropic, así que lo apuntamos a nuestro
servidor con `DEEPSEEK_BASE_URL`. Código: `src/harness_lab/dsh.py`.

Perfiles:

| Perfil | Tools |
|---|---|
| `sdk-minimal` | `bash`, `read_image`, `working_directory` |
| `sdk` | ficheros, búsqueda, skills (Office), plan, todo, subagentes, web... |

Empezamos con `sdk-minimal` y sin tools nuestras.

In [ ]:
from harness_lab import dsh

result = dsh.run_task(
    "¿Qué ficheros hay en el directorio actual? Usa bash.",
    mcp=False,
    session_id="demo-bash",
)
print("\n", result.finish_reason, "|", result.final_response)

¿Qué le ha mandado el harness al modelo? El servidor guarda cada petición.
Mirad el *system prompt* y la lista de tools: eso lo pone el harness, no vosotros.

In [ ]:
from harness_lab import traffic

traffic.show(api="messages");

### Añadir una tool propia: documentos Word por MCP

MCP (*Model Context Protocol*) es el estándar para conectar tools a un harness.
Nuestro servidor MCP (`src/harness_lab/tools/mcp_server.py`) ofrece:

- `create_word_document(filename, markdown, title)`: Markdown a `.docx` con python-docx.
- `read_word_document(filename)` y `list_workspace()`.
- `compare_checkpoints(base, edited)`: para la sección 6.

`dsh.py` lo registra con un *patch* de configuración. Dentro del harness aparecen como
`mcp__lab__create_word_document`. Codex y Claude Code usan exactamente el mismo servidor.

In [ ]:
result = dsh.run_task(
    "Crea un documento Word llamado harness.docx que explique qué es un harness de agentes. "
    "Incluye un título, una introducción, una lista de componentes y una tabla con tres harnesses "
    "(DeepSeek Harness, Codex, Claude Code). Usa la herramienta create_word_document.",
    session_id="demo-word",
)
print("\n", result.finish_reason, "|", result.final_response)

In [ ]:
from harness_lab.tools.docx_tools import read_word_document, list_workspace

print(list_workspace())
print(read_word_document("harness.docx"))

El documento está en `workspace/harness.docx`. Descargadlo desde code-server y abridlo en Word.

### El perfil completo

Con `profile="sdk"` el harness añade muchas más tools. Comparad el tamaño del system prompt
y la lista de tools con el perfil mínimo. Más tools = más contexto = más trabajo para un
modelo pequeño.

In [ ]:
result = dsh.run_task("Di hola en una frase.", profile="sdk", mcp=False, session_id="demo-sdk")
s = traffic.show(api="messages", system_preview=300)

## 5. Codex y Claude Code (opcional)

Son programas aparte. En la DGX se instalan sin root:

```bash
bash scripts/install_clis.sh
```

Cada uno recibe su propia configuración dentro del repo (`.codex_home/`, `.claude_home/`)
apuntando al servidor local, más nuestro servidor MCP. Código: `src/harness_lab/clis.py`.

In [ ]:
from harness_lab import clis

skip = os.environ.get("HARNESS_SKIP_CLIS") == "1"
if shutil.which("codex") and not skip:
    print(clis.run_codex("Crea un documento Word codex.docx con dos frases sobre Codex."))
    traffic.show(api="responses", system_preview=300)
else:
    print("codex no está instalado")

In [ ]:
if shutil.which("claude") and not skip:
    print(clis.run_claude("Crea un documento Word claude.docx con dos frases sobre Claude Code."))
    traffic.show(api="messages", system_preview=300)
else:
    print("claude no está instalado")

Mismo modelo, tres harnesses. Cambian el system prompt, las tools, los permisos y el bucle.
Por eso el mismo modelo se comporta distinto en cada uno.

## 6. Weight-space programming

[Weight-space programming](https://rikiyat.github.io/blog/weight-space-programming/)
(Takehi, Teoh, Zhang, Khattab · MIT, nov. 2026): un LLM frontera, dentro de un harness con un
kernel de Python, **lee y escribe pesos** de otras redes. Sin gradientes, sin pérdida.

Su caso con Qwen3.5-0.8B: el modelo entraba en bucles hasta agotar los tokens. El LLM localizó
una dirección escrita por las MLP de las capas 15 y 19 y la restó de 64 columnas en cada una
(131.072 pesos, 0,016 % del modelo). MATH-500 pasó de 7,9 % a 48,1 % según el post.

Relación con ARCA:

| | SFT / GRPO (ARCA) | Weight-space programming |
|---|---|---|
| Qué cambia | todos los pesos o un LoRA | unas pocas columnas |
| Cómo | gradientes sobre una pérdida o recompensa | un agente razona y edita |
| Señal | datos / recompensa verificable | pocas *forward passes* de diagnóstico |
| Límite | necesita datos y cómputo | quita obstáculos, no añade capacidades |

**Leer** la edición publicada: comparamos los dos checkpoints tensor a tensor.

In [ ]:
from harness_lab.wsp.diff import compare_checkpoints, format_report

if not config.SCRIPTED:
    diffs = compare_checkpoints(config.WSP_BASE_MODEL_ID, config.WSP_EDITED_MODEL_ID)
    print(format_report(diffs))

Si el informe dice `rank=1` y `matches 'remove u from columns'` cerca de 1, la edición es
exactamente "quitar una dirección `u` de esas columnas": `W[:, cols] -= u (uᵀ W[:, cols])`.

**Medir** el fallo: ¿cuántas respuestas se quedan en bucle? (GSM8K *train*, ~10 min en la DGX)

In [ ]:
from harness_lab.wsp.cli import load_model
from harness_lab.wsp.probe import load_practice, run_probe

RUN_WSP = not config.SCRIPTED   # cambiad a False si vais justos de tiempo
if RUN_WSP:
    server.terminate()           # liberamos la GPU del servidor
    wsp_model, wsp_tok = load_model(config.WSP_BASE_MODEL_ID)
    before = run_probe(wsp_model, wsp_tok, load_practice(16), max_new_tokens=2048)
    print(before.summary())

**Escribir** una edición propia con la misma forma (`src/harness_lab/wsp/edit.py`): la
dirección sale de la diferencia de activaciones entre respuestas en bucle y respuestas limpias.
Es una receta simple, no el razonamiento de un LLM frontera.

In [ ]:
from harness_lab.wsp.edit import apply_edit, find_direction
from harness_lab.wsp.probe import looks_looping

if RUN_WSP:
    bad = [o for o, t in zip(before.outputs, before.truncated) if t or looks_looping(o)]
    good = [o for o, t in zip(before.outputs, before.truncated) if not (t or looks_looping(o))]
    print(f"en bucle: {len(bad)}  limpias: {len(good)}")
    if len(bad) >= 2 and len(good) >= 2:
        u = find_direction(wsp_model, wsp_tok, bad, good, layer=15)
        edited = apply_edit(wsp_model, u, layers=[15, 19], n_columns=64)
        after = run_probe(wsp_model, wsp_tok, load_practice(16, seed=1), max_new_tokens=2048)
        print("antes: ", before.summary())
        print("después:", after.summary())

Con vuestro modelo de ARCA (Qwen3-0.6B + LoRA de GRPO) el comando es:

```bash
uv run harness-wsp probe --model Qwen/Qwen3-0.6B --adapter ../dgm-arca/rlm/weights/final_rlm_lora
uv run harness-wsp edit  --model Qwen/Qwen3-0.6B --adapter ../dgm-arca/rlm/weights/final_rlm_lora --layers 14 20
```

Y el paso que hace "weight-space programming" de verdad: darle estas herramientas a un agente
(DeepSeek Harness con un modelo grande, Codex o Claude Code) y pedirle que diagnostique y edite.

## 7. Tests

Los tests no necesitan GPU ni descargas: usan un modelo simulado detrás del servidor real y
ejecutan DeepSeek Harness de verdad.

In [ ]:
import subprocess
print(subprocess.run(["uv", "run", "pytest", "-q", "-k", "not notebook"], capture_output=True, text=True).stdout[-800:])

In [ ]:
# Al terminar: parar el servidor y liberar la GPU
if server.poll() is None:
    server.terminate()

## Ejercicios

1. Añadid una tool al servidor MCP (por ejemplo, `create_excel` con openpyxl o una tool de
   vuestro tema de ARCA) y usadla desde DeepSeek Harness.
2. Comparad en `outputs/traffic/` lo que envían DeepSeek Harness, Codex y Claude Code.
   ¿Cuántos tokens de system prompt y tools gasta cada uno antes de vuestra pregunta?
3. Repetid la tarea del Word con `thinking=True`. ¿Mejora? ¿Cuánto tarda?
4. Medid la tasa de bucles de vuestro modelo de ARCA con `harness-wsp probe` y probad una edición.